# Configuration env

In [1]:
from pathlib import Path
import os

# remonte jusqu'au dossier qui contient .git, puis s'y place
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").is_dir())
os.chdir(ROOT)
print("racine projet :", ROOT)

racine projet : /Users/benjaminscemama/dev/market-risk-control


In [2]:
%load_ext sql
%config SqlMagic.autopandas = True
%sql duckdb:///:memory:
%sql ATTACH IF NOT EXISTS 'data/risk.db' AS r (TYPE sqlite);

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

,Success


In [3]:
%%sql
-- Test configuration environnement
SELECT name FROM (SHOW ALL TABLES) WHERE database = 'r' ORDER BY name;

Running query in 'duckdb:///:memory:'

,name
0,mkt_forward_curve
1,mkt_spot_hourly
2,pos_snapshot
3,ref_contract
4,ref_customer
5,ref_site
6,trd_deal


# 1. `ref_customer`

## Niveau 0 - cadrage

In [12]:
%%sql 
-- Nombre de lignes
select count(*) as nb_raw from r.ref_customer;

Running query in 'duckdb:///:memory:'

,nb_raw
0,220


---

In [36]:
%%sql
-- unicité de customer_id
select count(*) as n_raw, count(distinct customer_id) as n_distinct, count(customer_id) as n_customer_id from r.ref_customer;

Running query in 'duckdb:///:memory:'

,n_raw,n_distinct,n_customer_id
0,220,220,220


---

In [17]:
%%sql 
-- Unicité customer_name
select customer_name, count(*) as nb_customer_name from r.ref_customer
group by customer_name
having count(*) > 1;

Running query in 'duckdb:///:memory:'

,customer_name,nb_customer_name


In [44]:
%%sql 
select upper(trim(customer_name)) as customer_name from r.ref_customer
group by upper(trim(customer_name))
having count(*) > 1;

Running query in 'duckdb:///:memory:'

,customer_name


In [56]:
%%sql
with n as ( 
    select 
        customer_name as n0,
        upper(trim(customer_name)) as n1,
        replace(upper(trim(customer_name)), '.', '') as n2,
        regexp_replace(upper(trim(customer_name)), '\s+', ' ', 'g') as n3,
        regexp_replace(strip_accents(upper(trim(customer_name))), '[^A-Z0-9]', '', 'g') as n4
    from r.ref_customer
)
select 
    count(*) as lignes,
    count(distinct n0) as brut,
    count(distinct n1) as upper_trim,
    count(distinct n2) as sans_point,
    count(distinct n3) as espaces_normalises,
    count(distinct n4) as alphanumerique_seul
from n;


Running query in 'duckdb:///:memory:'

,lignes,brut,upper_trim,sans_point,espaces_normalises,alphanumerique_seul
0,220,220,220,220,220,220


---

In [23]:
%%sql
-- Exhaustivité « tout customer_id référencé dans ref_site ou ref_contract figure ici »
select rs.customer_id from r.ref_site as rs
left join r.ref_customer as rc on rs.customer_id = rc.customer_id
where rc.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


In [25]:
%%sql
-- Exhaustivité « tout customer_id référencé dans ref_site ou ref_contract figure ici »
select rco.customer_id from r.ref_contract as rco
left join r.ref_customer as rc on rco.customer_id = rc.customer_id
where rc.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


---

In [33]:
%%sql 
-- Tous les clients ont un site
select rc.customer_id from r.ref_customer as rc
left join r.ref_site as rs on rc.customer_id = rs.customer_id
where rs.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


In [34]:
%%sql
-- Tous les clients ont un contrat
select rc.customer_id from r.ref_customer as rc
left join r.ref_contract as rco on rc.customer_id = rco.customer_id
where rco.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id
0,C100035
1,C100132
2,C100046
3,C100103
4,C100116
...,...
69,C100152
70,C100194
71,C100013
72,C100076


---